# Conversión de imágenes de frutas a CSV (escala de grises 0-255)

Genera una **matriz** donde cada fila es una imagen en escala de grises
redimensionada a `88x88 = 7744` píxeles y aplanada, más su etiqueta.

- Encabezado **solo numérico** (`0` .. `7744`), sin letras ni columnas extra.
- Píxeles en gris con **enteros de 0 a 255** (igual que `frutas5.csv`).
- Etiqueta numérica en la última columna (`7744`).
- Sin columna de índice y sin nombres de texto.


In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd

ZIP_PATH = "dataset.zip"
EXTRACCION = "dataset_extracted"
ANCHO, ALTO = 88, 88
PIXELES = ANCHO * ALTO
SALIDA_CSV = "frutas_gris.csv"

# Lector de imágenes en escala de grises.
# Usa OpenCV si está instalado; si no, TensorFlow.
try:
    import cv2
    def leer_gris(datos):
        arr = np.frombuffer(datos, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        return cv2.resize(img, (ANCHO, ALTO))
    print("Usando OpenCV para leer imágenes.")
except ImportError:
    import tensorflow as tf
    tf.get_logger().setLevel("ERROR")
    def leer_gris(datos):
        img = tf.io.decode_image(datos, channels=1, expand_animations=False)
        img = tf.image.resize(img, [ALTO, ANCHO])
        return np.asarray(img, dtype=np.uint8)[:, :, 0]
    print("OpenCV no disponible: usando TensorFlow para leer imágenes.")


In [1]:
# 1) Descomprimir el zip
if not os.path.isdir(os.path.join(EXTRACCION, "fruits")):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACCION)
    print("Zip extraído en:", EXTRACCION)

RUTA_FRUITS = os.path.join(EXTRACCION, "fruits")
categorias = sorted(c for c in os.listdir(RUTA_FRUITS)
                    if os.path.isdir(os.path.join(RUTA_FRUITS, c)))
mapeo = {cat: float(i) for i, cat in enumerate(categorias)}
print(f"{len(categorias)} categorías de frutas.")
print("Mapeo de etiquetas:", mapeo)


NameError: name 'os' is not defined

In [ ]:
# 2) Recorrer carpeta por carpeta e imagen por imagen -> vector por imagen
imagenes = []
for cat in categorias:
    ruta_cat = os.path.join(RUTA_FRUITS, cat)
    for nombre in sorted(os.listdir(ruta_cat)):
        if nombre.lower().endswith((".jpg", ".jpeg", ".png")):
            imagenes.append((cat, os.path.join(ruta_cat, nombre)))
print("Total de imágenes encontradas:", len(imagenes))

# Matriz de píxeles (enteros 0-255) y vector de etiquetas
matriz_pixeles = np.empty((len(imagenes), PIXELES), dtype=np.uint8)
etiquetas = np.empty((len(imagenes),), dtype=np.float32)
fila = 0
for cat, ruta in imagenes:
    with open(ruta, "rb") as fh:
        img = leer_gris(fh.read())
    if img is None:
        continue
    matriz_pixeles[fila] = img.reshape(-1)   # gris 88x88 -> 7744 enteros 0-255
    etiquetas[fila] = mapeo[cat]
    fila += 1

matriz_pixeles = matriz_pixeles[:fila]
etiquetas = etiquetas[:fila]
print("Forma de la matriz (filas, columnas):", (fila, PIXELES + 1))


In [ ]:
# 3) Construir el DataFrame con encabezado SOLO numérico y exportar
columnas_pix = [str(i) for i in range(PIXELES)]      # '0' ... '7743'
df = pd.DataFrame(matriz_pixeles, columns=columnas_pix)
df[str(PIXELES)] = etiquetas                         # columna '7744' = etiqueta

display(df.head())
print("Forma:", df.shape)
print("Últimas columnas:", list(df.columns[-5:]))

# index=False -> sin columna extra; encabezado 0..7744; sin letras
df.to_csv(SALIDA_CSV, index=False)
print(f"[OK] Guardado '{SALIDA_CSV}' con {len(df)} filas y {df.shape[1]} columnas.")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from sklearn.model_selection import train_test_split

# ============================================================
# CARGAR Y PREPARAR DATASET
# ============================================================

df = pd.read_csv("frutas_gris.csv")

# Eliminar filas que contengan valores nulos (NaN)
df.dropna(inplace=True)

X = df.drop("7744", axis=1).values
y = df["7744"].values

X = X.astype("float32") / 255.0

# 7744 píxeles → imagen de 88x88 en escala de grises
X = X.reshape(-1, 88, 88, 1)

# Las etiquetas ya vienen indexadas de 0 a 32 (33 frutas), NO restamos 1
NUM_CLASES = 33

# ============================================================
# DIVISIÓN DEL DATASET
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ============================================================
# RED CONVOLUCIONAL
# ============================================================

modelo = Sequential()

modelo.add(Input(shape=(88, 88, 1)))

# ---------- NIVEL CONVOLUCIONAL 1 ----------
modelo.add(Conv2D(32, (3, 3), activation="relu"))
modelo.add(MaxPooling2D((2, 2)))

# ---------- NIVEL CONVOLUCIONAL 2 ----------
modelo.add(Conv2D(64, (5, 5), activation="relu"))
modelo.add(MaxPooling2D((2, 2)))

# ---------- NIVEL CONVOLUCIONAL 3 ----------
modelo.add(Conv2D(128, (3, 3), activation="relu"))
modelo.add(MaxPooling2D((2, 2)))

# ============================================================
# RED NEURONAL
# ============================================================

modelo.add(Flatten())
modelo.add(Dense(256, activation="relu"))
modelo.add(Dense(128, activation="relu"))
modelo.add(Dense(64, activation="relu"))

# ============================================================
# CAPA DE SALIDA
# ============================================================

# 33 neuronas = 33 frutas. Sin Softmax para observar las salidas.
modelo.add(Dense(NUM_CLASES))

# ============================================================
# RESUMEN DEL MODELO
# ============================================================

modelo.summary()

# ============================================================
# COMPILACIÓN
# ============================================================

modelo.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# ============================================================
# ENTRENAMIENTO
# ============================================================

historial = modelo.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.20
)

# ============================================================
# EVALUACIÓN
# ============================================================

loss, accuracy = modelo.evaluate(X_test, y_test, verbose=0)
print(f"\nPrecisión: {accuracy * 100:.2f}%")

# ============================================================
# PREDICCIONES
# ============================================================

salidas = modelo.predict(X_test, verbose=0)

# ============================================================
# MOSTRAR LAS SALIDAS
# ============================================================

for i in range(10):
    clase_predicha = np.argmax(salidas[i])   # ya está en 0..32
    clase_real = y_test[i]
    print(f"\nImagen {i + 1}")
    print("Salidas:", salidas[i])
    print("Predicción:", clase_predicha)
    print("Real:", clase_real)

# ============================================================
# GUARDAR MODELO
# ============================================================

modelo.save("clasificador_frutas_Brayan.keras")